# Lecture 6 — Spatial Heterogeneity and Calibration
## 第六讲 —— 空间异质性与校准

**Computational Methods for Heterogeneous-Agent Macro**
**异质性主体宏观的计算方法**

Jeffrey Sun

### Environment
### 运行环境

Activate the project, load `HouseholdStages` plus `Printf` / `Plots`.

激活项目，加载 `HouseholdStages` 以及 `Printf` 和 `Plots`。

In [ ]:
using Pkg
Pkg.activate("..")
Pkg.instantiate()
using HouseholdStages
using Printf
using Plots

## 1 · The spatial setup
## 1 · 空间设定

Three locations `loc1`, `loc2`, `loc3` with productivities $A_1 > A_2 > A_3$. Each location produces a single, costlessly-tradable, perfectly-substitutable numeraire good with Cobb–Douglas technology
$$Y_j = A_j K_j^{\alpha} L_j^{1-\alpha}.$$
Capital flows freely across locations and with the rest of the world, so the country is small in the world bond market: **$r$ is exogenous**. Within the country, MPK equalization across locations pins $K_j / L_j$ given $(r, A_j)$, and wages then satisfy
$$w_j = (1-\alpha)\, A_j\, \big(\tfrac{\alpha A_j}{r+\delta}\big)^{\alpha/(1-\alpha)}.$$

三个地点 `loc1`、`loc2`、`loc3`，生产率 $A_1 > A_2 > A_3$。每地用 Cobb–Douglas 技术生产同一种无成本可贸易、完全可替代的计价品 $Y_j = A_j K_j^{\alpha} L_j^{1-\alpha}$。资本可在地点间及与世界自由流动，本国在世界债券市场上规模小，因此 **$r$ 外生**。国内地点间 MPK 均等化，由 $(r, A_j)$ 决定 $K_j / L_j$，从而工资 $w_j = (1-\alpha) A_j \big(\alpha A_j/(r+\delta)\big)^{\alpha/(1-\alpha)}$。

Households are heterogeneous in **wealth** and **location** only (no idiosyncratic income shocks). Within a period:

家庭只在**财富**和**地点**两个维度上异质（不含个体收入冲击）。一期内分为三个阶段：

1. **Migration** — draw a Gumbel taste shock over destinations and pick a new location, paying the migration cost.  迁移：在目的地之间抽取 Gumbel 偏好冲击，付迁移成本后选择新地点。
2. **Wealth update** — receive the destination's wage and roll wealth $b \mapsto (1+r)b + w_j$.  财富更新：拿到目的地工资，财富按 $(1+r)b + w_j$ 演化。
3. **Consumption / savings** — pick next-period wealth on the grid.  消费/储蓄：在财富网格上选择下期财富。

The chain is
$$\text{Migration} \circ_s \text{WealthChange} \circ_s \text{ConsumptionSavings}.$$

链为 $\text{Migration} \circ_s \text{WealthChange} \circ_s \text{ConsumptionSavings}$。

### Parameters and the target shares
### 参数与目标份额

Standard macro calibration. The world rate is $r = 0.03$. Productivities are $(1.20, 1.00, 0.85)$ — `loc1` is the high-wage region. Migration cost $C[i, j]$ is symmetric, with adjacent locations cheaper to move between than distant ones.

The target population shares `s_data = (0.30, 0.30, 0.40)` are synthetic: the *data* say 40% of households live in `loc3` (the lowest-productivity region), so we will need a positive preference shifter on `loc3` to rationalize them. The whole point of the calibration in §6 is to recover that shifter.

标准宏观参数。世界利率 $r = 0.03$。生产率 $(1.20, 1.00, 0.85)$ —— `loc1` 是高工资地区。迁移成本 $C[i, j]$ 对称，相邻地点之间的成本低于较远地点。

目标人口份额 `s_data = (0.30, 0.30, 0.40)` 为合成数据：*数据*显示 40% 家庭住在 `loc3`（生产率最低的地区），要解释这一点需要在 `loc3` 上添加正的偏好平移。第 6 节的校准的全部意义就在于恢复这个平移。

In [ ]:
@kwdef struct SpatialParams3
    β :: Float64 = 0.96
    σ :: Float64 = 1.5
    α :: Float64 = 0.36
    δ :: Float64 = 0.08

    r :: Float64 = 0.03                       # world interest rate (exogenous)

    A :: NTuple{3,Float64} = (1.20, 1.00, 0.85)  # productivities

    # Preference shifters: +α_pref[j] added to the destination utility.
    # Absorbed into the migration cost matrix as column subtractions.
    α_pref :: Vector{Float64} = [0.0, 0.0, 0.0]

    # Migration cost matrix C[i, j] (origin → destination).
    C_base :: Matrix{Float64} = [0.0 0.5 1.0;
                                 0.5 0.0 0.5;
                                 1.0 0.5 0.0]

    ε_logit :: Float64 = 1.5                  # Gumbel scale

    N_w   :: Int     = 250
    w_min :: Float64 = 0.0
    w_max :: Float64 = 25.0
end
Base.Broadcast.broadcastable(p::SpatialParams3) = Ref(p)

const s_data = [0.30, 0.30, 0.40]
p = SpatialParams3()
@printf "β=%.2f, σ=%.2f, α=%.2f, δ=%.2f, r=%.4f\n" p.β p.σ p.α p.δ p.r
@printf "A = (%.2f, %.2f, %.2f);  ε_logit = %.2f\n" p.A[1] p.A[2] p.A[3] p.ε_logit
println("target shares = ", s_data)

## 2 · Wages from the world rate
## 2 · 由世界利率决定的工资

`spatial_wage(r, A_j, p)` implements $w_j = (1-\alpha) A_j (\alpha A_j / (r+\delta))^{\alpha/(1-\alpha)}$. `make_env(p)` packages $(r, w_1, w_2, w_3)$ for the household block to read.

`spatial_wage(r, A_j, p)` 实现 $w_j = (1-\alpha) A_j (\alpha A_j / (r+\delta))^{\alpha/(1-\alpha)}$。`make_env(p)` 把 $(r, w_1, w_2, w_3)$ 打包给家庭块读取。

In [ ]:
spatial_wage(r, A_j, p) = (1 - p.α) * A_j *
    (p.α * A_j / (r + p.δ))^(p.α / (1 - p.α))

make_env(p) = (; r = p.r,
                 w1 = spatial_wage(p.r, p.A[1], p),
                 w2 = spatial_wage(p.r, p.A[2], p),
                 w3 = spatial_wage(p.r, p.A[3], p))

env0 = make_env(p)
@printf "Wages: (w1, w2, w3) = (%.4f, %.4f, %.4f)\n" env0.w1 env0.w2 env0.w3
@printf "Ratio w1/w3 = %.3f (productivity ratio A1/A3 = %.3f)\n" env0.w1/env0.w3 p.A[1]/p.A[3]

## 3 · The household block
## 3 · 家庭块

Three stages, with the migration cost matrix encoding the **preference shifters** $\alpha_j$ as a column subtraction:
$$C_{\text{eff}}[i, j] = C_{\text{base}}[i, j] - \alpha_j.$$
This is the trick that lets us tune choice probabilities without changing the `MigrationStage` API — adding $\alpha_j$ to the destination utility and subtracting it from the destination cost are the same operation inside the log-sum-exp.

三个阶段。把**偏好平移** $\alpha_j$ 通过对迁移成本矩阵作列减法编码：$C_{\text{eff}}[i, j] = C_{\text{base}}[i, j] - \alpha_j$。在对数-求和-指数运算内部，把 $\alpha_j$ 加到目的地效用上、或从目的地成本中减去，是同一件事 —— 这样就无需修改 `MigrationStage` 的 API。

In [ ]:
_u_crra(c, ::Val{1})           = log(c)
_u_crra(c, ::Val{σv}) where σv = (c^(1 - σv)) / (1 - σv)
u_crra(c, valσ::Val) = c < 0 ? -Inf : _u_crra(c, valσ)

function effective_cost(p::SpatialParams3)
    C = similar(p.C_base)
    for i in 1:3, j in 1:3
        C[i, j] = p.C_base[i, j] - p.α_pref[j]
    end
    return C
end

function spatial_household(p::SpatialParams3)
    layout = StateLayout(
        StateAxis(:wealth,   continuous_grid(p.w_min, p.w_max;
                                             length = p.N_w, spacing = :log)),
        StateAxis(:location, categorical([:loc1, :loc2, :loc3])),
    )

    migration = MigrationStage(layout;
        location_axis  = :location,
        migration_cost = effective_cost(p),
        ε              = p.ε_logit,
    )

    receipt = WealthChangeStage(layout;
        wealth_post = function (cell; env)
            w_loc = cell.location == :loc1 ? env.w1 :
                    cell.location == :loc2 ? env.w2 :
                                              env.w3
            return (1 + env.r) * cell.wealth + w_loc
        end,
        wealth_axis = :wealth,
    )

    savings = ConsumptionSavingsStage(layout;
        β               = p.β,
        utility         = (cell, c; env) -> u_crra(c, Val(p.σ)),
        wealth_axis     = :wealth,
        monotone_search = :divide_conquer,
    )

    return lift_moments(migration ∘ₛ receipt ∘ₛ savings;
        K_total = at_end(integrand = (cell; env) -> cell.wealth,                        reduce = sum),
        L1      = at_end(integrand = (cell; env) -> cell.location == :loc1 ? 1.0 : 0.0, reduce = sum),
        L2      = at_end(integrand = (cell; env) -> cell.location == :loc2 ? 1.0 : 0.0, reduce = sum),
        L3      = at_end(integrand = (cell; env) -> cell.location == :loc3 ? 1.0 : 0.0, reduce = sum),
    )
end

hh = spatial_household(p)
buffers = allocate(hh)
dims = layout_size(first(hh.stages).input_layout)
@printf "Layout: wealth %d × location %d = %d cells\n" dims[1] dims[2] prod(dims)

## 4 · Solving the household at $\alpha = 0$
## 4 · 在 $\alpha = 0$ 时求解家庭块

With *no* preference shifters, choice probabilities depend only on wages and the migration cost. `loc1` has the highest wage; we expect it to attract the largest population share.

无偏好平移时，选择概率只取决于工资和迁移成本。`loc1` 工资最高，预期吸引最大份额人口。

In [ ]:
function solve_household(hh, buffers, p; V_init = nothing, Λ_init = nothing)
    env = make_env(p)
    res = isnothing(V_init) ?
        solve_steady_state_given_env!(hh, env, buffers) :
        solve_steady_state_given_env!(hh, env, buffers;
                                       V_init = V_init, Λ_init = Λ_init)
    m = compute_moments(hh, env)
    shares = [m.L1, m.L2, m.L3] ./ (m.L1 + m.L2 + m.L3)
    return (; V = res.V, Λ = res.Λ, env, moments = m, shares,
              vfi_iters = res.vfi_iters, lambda_iters = res.lambda_iters)
end

out0 = solve_household(hh, buffers, p)
@printf "VFI %d iters, Λ %d iters\n" out0.vfi_iters out0.lambda_iters
@printf "Population shares (uncalibrated): (%.3f, %.3f, %.3f)\n" out0.shares[1] out0.shares[2] out0.shares[3]
@printf "Total wealth K_total = %.4f\n" out0.moments.K_total

The high-productivity region `loc1` is overpopulated relative to the data; `loc3` is underpopulated. Something is keeping people in `loc3` that the model — with $\alpha = 0$ — does not see. That something is what we will calibrate.

高生产率地区 `loc1` 在模型中的人口超过数据；`loc3` 不足。某个让人留在 `loc3` 的因素，在 $\alpha = 0$ 的模型中是看不见的 —— 这正是我们要校准的对象。

In [ ]:
xs = 1:3
plt_init = plot(title = "Uncalibrated vs target shares", ylabel = "share",
                xticks = (xs, ["loc1", "loc2", "loc3"]),
                ylims = (0.0, 0.5), legend = :topright, size = (640, 360))
bar!(plt_init, xs .- 0.18, out0.shares; bar_width = 0.30, label = "model (α=0)")
bar!(plt_init, xs .+ 0.18, s_data;       bar_width = 0.30, label = "target (data)")

## 5 · Calibration by Berry contraction
## 5 · Berry 收缩校准

We want $\alpha = (\alpha_1, \alpha_2, \alpha_3)$ such that the model's stationary population shares equal `s_data`. In static logit, choice probabilities satisfy $\log s_j = (\delta_j - \bar\delta)/\varepsilon + \text{const}$, where $\delta_j$ contains $\alpha_j$ additively. **Berry (1994)** proved this map admits a contraction inversion:
$$\alpha_j^{(k+1)} = \alpha_j^{(k)} + \varepsilon \cdot \bigl(\log s_j^{\text{data}} - \log s_j^{\text{model}}\bigr).$$
The model here is dynamic — $V_j$ depends on $\alpha$ through expected future migration value — so the contraction is no longer exact. Empirically it still works for this regularity: log-shares are monotone in $\alpha$, and small steps in $\varepsilon$-metric stay stable. (For the strict theorem, see Berry, Levinsohn, Pakes 1995, p. 854.)

我们要找 $\alpha = (\alpha_1, \alpha_2, \alpha_3)$，使模型平稳份额等于 `s_data`。静态 logit 中，选择概率满足 $\log s_j = (\delta_j - \bar\delta)/\varepsilon + \text{const}$，其中 $\delta_j$ 加性包含 $\alpha_j$。**Berry (1994)** 证明该映射存在收缩反演：$\alpha_j^{(k+1)} = \alpha_j^{(k)} + \varepsilon \bigl(\log s_j^{\text{data}} - \log s_j^{\text{model}}\bigr)$。这里的模型是动态的 —— $V_j$ 通过未来迁移期望值依赖于 $\alpha$ —— 严格收缩不再成立，但只要 $\alpha$ 与 log-份额单调相关、且步长以 $\varepsilon$ 为度量，实证上仍可收敛。严格定理见 Berry, Levinsohn, Pakes (1995, p. 854)。

**Note on terminology.** What we are doing is sometimes called *indirect inference* in macro talk because we infer $\alpha$ (unobserved) by matching moments through the model. Strictly, Gourieroux–Monfort–Renault (1993) "indirect inference" uses an *auxiliary* model; here we use the model itself as its own auxiliary. The looser usage is common.

**术语说明。**这里所做的有时被称为*间接推断*（indirect inference），因为我们通过模型匹配矩来推断不可观测的 $\alpha$。严格的 Gourieroux–Monfort–Renault (1993) 间接推断使用*辅助模型*，这里以模型自身充当辅助。这种较宽松的用法在宏观文献中常见。

In [ ]:
function set_shifters!(hh, p::SpatialParams3, α_new::AbstractVector)
    mig = hh.stages[1]
    for i in 1:3, j in 1:3
        mig.migration_cost[i, j] = p.C_base[i, j] - α_new[j]
    end
    return hh
end

function calibrate_shifters!(hh, buffers, p, s_data;
                              damping   = 1.0,
                              tol       = 5e-3,
                              maxiter   = 30,
                              verbosity = 1)
    α = copy(p.α_pref)
    V, Λ = nothing, nothing
    α_history   = Vector{Float64}[copy(α)]
    gap_history = Vector{Float64}[]
    last_out    = nothing
    iters       = 0
    converged   = false

    while iters < maxiter
        set_shifters!(hh, p, α)
        out = solve_household(hh, buffers, p; V_init = V, Λ_init = Λ)
        last_out = out
        V, Λ = out.V, out.Λ

        gap = log.(s_data) .- log.(max.(out.shares, 1e-8))
        push!(gap_history, copy(gap))
        iters += 1

        verbosity > 0 && @printf(
            "  iter %2d: shares = (%.3f, %.3f, %.3f); α = (%.3f, %.3f, %.3f); ‖gap‖∞ = %.4f\n",
            iters, out.shares[1], out.shares[2], out.shares[3],
            α[1], α[2], α[3], maximum(abs.(gap)))

        if maximum(abs.(gap)) < tol
            converged = true
            break
        end

        # Berry contraction: step by ε · log-share gap. Normalize α[1] = 0.
        α .+= damping * p.ε_logit .* gap
        α .-= α[1]
        push!(α_history, copy(α))
    end

    return (; α, iters, converged, α_history, gap_history,
              shares = last_out.shares, V = last_out.V, Λ = last_out.Λ,
              env = last_out.env, moments = last_out.moments)
end

### Run the calibration
### 运行校准

The loop starts at $\alpha = 0$ (where `loc3` is underpopulated), takes a Berry step, re-solves the household, repeats. Convergence is geometric.

迭代从 $\alpha = 0$ 开始（此时 `loc3` 人口不足），作 Berry 步，重新求解家庭块，重复。收敛速率几何级。

In [ ]:
calib = calibrate_shifters!(hh, buffers, p, s_data; verbosity = 1)
@printf "\nFinal α = (%.4f, %.4f, %.4f); converged = %s in %d iterations\n" calib.α[1] calib.α[2] calib.α[3] calib.converged calib.iters
@printf "Final shares: (%.3f, %.3f, %.3f) vs target (%.3f, %.3f, %.3f)\n" calib.shares[1] calib.shares[2] calib.shares[3] s_data[1] s_data[2] s_data[3]

### Convergence diagnostics
### 收敛诊断

The log-share gap collapses geometrically; the shifter trajectory overshoots once on iter 2 (when the contraction is far from the fixed point) and damps in.

log 份额差呈几何级衰减；偏好平移轨迹在第 2 步过冲一次（此时离不动点较远），随后阻尼收敛。

In [ ]:
αs   = hcat(calib.α_history...)
gaps = hcat(calib.gap_history...)

plt_α   = plot(1:size(αs, 2), αs',
               label = ["α[1]" "α[2]" "α[3]"],
               marker = :circle, linewidth = 2,
               xlabel = "calibration iteration", ylabel = "α_j",
               title = "Preference shifter trajectory")

plt_gap = plot(1:size(gaps, 2), maximum.(abs, eachcol(gaps)),
               marker = :circle, linewidth = 2, yaxis = :log,
               xlabel = "calibration iteration", ylabel = "max gap (log scale)",
               title = "‖log s_data − log s_model‖∞", legend = false)

plot(plt_α, plt_gap; layout = (1, 2), size = (900, 360))

### Final shares: uncalibrated, calibrated, target
### 最终份额：未校准 / 校准 / 目标

In [ ]:
xs = 1:3
plt_final = plot(title = "Population shares", ylabel = "share",
                 xticks = (xs, ["loc1", "loc2", "loc3"]),
                 ylims = (0.0, 0.5), legend = :topright, size = (640, 360))
bar!(plt_final, xs .- 0.25, out0.shares;  bar_width = 0.22, label = "uncalibrated (α=0)")
bar!(plt_final, xs,         calib.shares; bar_width = 0.22, label = "calibrated")
bar!(plt_final, xs .+ 0.25, s_data;       bar_width = 0.22, label = "target (data)")

## 6 · Reading the calibrated shifters
## 6 · 解读校准的偏好平移

- $\alpha_1 = 0$ by normalization. `loc1`'s 30% share in the data is *fully explained* by its productivity advantage; it needs no amenity boost.
- $\alpha_2 \approx 0$ (about $-0.013$). `loc2`'s 30% share is also broadly consistent with the wage-only model.
- $\alpha_3 \approx 0.75$. `loc3` houses 40% of households despite having the *lowest* wage. The data are telling us about $0.75$ units of utility per period of "amenity" or unobserved preference that the wage model misses. This is the kind of inference indirect identification gives us: a value for a quantity we cannot see directly.

- $\alpha_1 = 0$（标准化）。数据中 `loc1` 的 30% 份额*完全*由其生产率优势解释，无需便利度加成。
- $\alpha_2 \approx 0$（约 $-0.013$）。`loc2` 的 30% 份额也基本与只看工资的模型一致。
- $\alpha_3 \approx 0.75$。`loc3` 的工资*最低*，却仍占 40% 人口。数据告诉我们：每期约 $0.75$ 个单位的效用，对应工资模型看不到的便利度或未观测偏好。这就是间接识别得到的推断 —— 一个我们无法直接观测的量被给了赋值。

## 7 · A closer look: wealth distribution by location
## 7 · 进一步观察：按地点的财富分布

The stationary $\Lambda$ is a $(N_w \times 3)$ array. Marginalizing along wealth at each location gives the population share; slicing instead by wealth shows how heterogeneous savers concentrate. With no income heterogeneity, the wealth distribution is much tighter than Aiyagari's — the only buffer-stock motive comes from the *possibility* of moving to a different wage in the future.

平稳分布 $\Lambda$ 是一个 $(N_w \times 3)$ 数组。沿财富维度求和得到每地点的人口份额；沿地点维度切片则展示了不同储蓄者的集中度。由于没有收入异质性，财富分布比 Aiyagari 紧很多 —— 唯一的预防性储蓄动机来自*未来可能*迁到不同工资的地点。

In [ ]:
wgrid = first(hh.stages).input_layout.axes[1].kind.grid
Λ_norm = calib.Λ
plot(wgrid, Λ_norm;
     labels = ["loc1" "loc2" "loc3"],
     xlims = (0, 5),
     xlabel = "wealth", ylabel = "mass",
     title = "Stationary wealth distribution by location",
     linewidth = 2, size = (720, 360))

## 8 · Foreshadow
## 8 · 预告

Today's bond market was exogenous: we set $r$ from the world. **L07** brings the asset market back inside the model and adds *aggregate* uncertainty — the Krusell–Smith problem. The migration toolkit you just used will reappear in L08–L10 as one instance of a more general pattern (discrete choice as a stage, with calibration as an outer loop).

今天的债券市场外生：我们从世界利率取 $r$。**第 7 讲** 将资产市场重新纳入模型，并加入*总量*不确定性 —— Krusell–Smith 问题。你刚刚使用的迁移工具将在 L08–L10 中作为更普遍模式（"离散选择作为一个 stage、校准作为一外层循环"）的一种实例重新出现。